 # Milestone 2: Data Quality dan Profiling
 

In [1]:
from pathlib import Path
import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT.parent / "data" / "raw" / "apbd_indonesia_2011_2026.parquet"
OUTPUT_PATH = PROJECT_ROOT.parent / "data" / "processed" / "apbd_indonesia_bersih.parquet"

if not DATA_PATH.exists():
    DATA_PATH = Path("../data/raw/apbd_indonesia_2011_2026.parquet")
    OUTPUT_PATH = Path("../data/processed/apbd_indonesia_bersih.parquet")

df = pl.read_parquet(DATA_PATH)
print(f"Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom")
df.head()

Ukuran awal: 3,132,863 baris x 11 kolom


tahun,bulan,id_provinsi,provinsi,id_pemda,pemda,no_akun,akun,anggaran,realisasi,persentase
i16,i8,str,str,str,str,i16,str,f64,f64,f64
2011,1,"""01""","""Provinsi Aceh""","""00""","""Prov. Aceh""",1,"""Pendapatan""",7.0894e12,7.6093e12,107.33
2011,1,"""01""","""Provinsi Aceh""","""00""","""Prov. Aceh""",2,"""PAD""",7.9728e11,8.0284e11,100.7
2011,1,"""01""","""Provinsi Aceh""","""00""","""Prov. Aceh""",3,"""Pajak Daerah""",6.2271e11,5.8618e11,94.13
2011,1,"""01""","""Provinsi Aceh""","""00""","""Prov. Aceh""",4,"""Retribusi Daerah""",1.3264e10,9.4160e9,70.99
2011,1,"""01""","""Provinsi Aceh""","""00""","""Prov. Aceh""",5,"""Hasil Pengelolaan Kekayaan Dae…",1.0200e11,6.6007e10,64.71


# 1. Profiling dengan DuckDB SUMMARIZE

In [2]:
connection = duckdb.connect()
connection.register("dataset", df)

profil = connection.execute("SUMMARIZE SELECT * FROM dataset").pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""tahun""","""SMALLINT""","""2011""","""2026""",19,"""2018.3805592520323""","""4.510928324219804""","""2014""","""2019""","""2022""",3132863,0.00
"""bulan""","""TINYINT""","""1""","""12""",13,"""6.432941689438702""","""3.4319052392576586""","""3""","""6""","""9""",3132863,0.00
"""id_provinsi""","""VARCHAR""","""01""","""38""",37,null,null,null,null,null,3132863,0.00
"""provinsi""","""VARCHAR""","""Provinsi Aceh""","""Provinsi Sumatera Utara""",41,null,null,null,null,null,3132863,0.00
"""id_pemda""","""VARCHAR""","""00""","""38""",37,null,null,null,null,null,3132863,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""no_akun""","""SMALLINT""","""1""","""43""",43,"""16.736300629807303""","""9.732570264531374""","""8""","""16""","""24""",3132863,0.00
"""akun""","""VARCHAR""","""Bagi Hasil Provinsi atau Pemda…","""TKDD""",66,null,null,null,null,null,3132863,0.00
"""anggaran""","""DOUBLE""","""-7117387800000.0""","""82663062617784.0""",156853,"""387306527202.41235""","""1391048613924.274""","""6198417021.462367""","""61896387837.55093""","""358367063774.4718""",3132863,0.00


# 2. DETEKSI MISSING VALUES

In [3]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns(
        (pl.col("jumlah_null") / jumlah_baris * 100).round(2).alias("persen_null")
    )
    .filter(pl.col("jumlah_null") > 0)
    .sort("jumlah_null", descending=True)
)

if len(null_summary) == 0:
    print("Tidak ditemukan missing values (null) pada seluruh kolom dataset.")
else:
    display(null_summary)

Tidak ditemukan missing values (null) pada seluruh kolom dataset.


# 3. PENANGANAN MISSING VALUES

In [4]:
df_missing_handled = df.with_columns(
    pl.when(pl.col("anggaran") > 0)
    .then((pl.col("realisasi") / pl.col("anggaran")) * 100)
    .otherwise(0.0)
    .alias("persentase")
)

print(f"Null sebelum: {df.null_count().sum_horizontal().sum():,}")
print(f"Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}")

Null sebelum: 0
Null sesudah: 0


# 4. DETEKSI DAN PENANGANAN OUTLIER / ANOMALI

In [5]:
PERSENTASE_MIN = 0.0
PERSENTASE_MAX = 200.0

df_outlier_handled = df_missing_handled.with_columns(
    pl.col("persentase").clip(lower_bound=PERSENTASE_MIN, upper_bound=PERSENTASE_MAX),
    pl.col("anggaran").clip(lower_bound=0.0),
    pl.col("realisasi").clip(lower_bound=0.0),
)

# 5. DEDUPLIKASI

In [6]:
KUNCI_BISNIS = ["tahun", "bulan", "id_provinsi", "id_pemda", "no_akun"]

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled.select(KUNCI_BISNIS).n_unique()

print(f"Total baris: {total_baris:,}")
print(f"Baris unik berdasarkan {KUNCI_BISNIS}: {baris_unik:,}")
print(f"Duplikat: {total_baris - baris_unik:,}")

df_dedup = df_outlier_handled.unique(
    subset=KUNCI_BISNIS,
    keep="first",
    maintain_order=True,
)

Total baris: 3,132,863
Baris unik berdasarkan ['tahun', 'bulan', 'id_provinsi', 'id_pemda', 'no_akun']: 3,132,863
Duplikat: 0


# 6. VALIDASI DATA DASAR (DUCKDB)

In [7]:
connection.register("dataset_bersih", df_dedup)

validasi = connection.execute("""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN anggaran IS NULL OR realisasi IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN anggaran < 0 OR realisasi < 0 THEN 1 ELSE 0 END) AS pelanggaran_nilai_negatif,
        SUM(CASE WHEN bulan < 1 OR bulan > 12 THEN 1 ELSE 0 END) AS pelanggaran_bulan,
        SUM(CASE WHEN persentase < 0 OR persentase > 200 THEN 1 ELSE 0 END) AS pelanggaran_persentase
    FROM dataset_bersih
""").pl()
validasi

total_baris,pelanggaran_null,pelanggaran_nilai_negatif,pelanggaran_bulan,pelanggaran_persentase
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
3132863,0,0,0,0


# 7. SIMPAN DATA BERSIH

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f"Baris sebelum cleaning: {len(df):,}")
print(f"Baris sesudah cleaning: {len(df_dedup):,}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")

Baris sebelum cleaning: 3,132,863
Baris sesudah cleaning: 3,132,863
Data bersih tersimpan di: /home/jovyan/work/data/processed/apbd_indonesia_bersih.parquet


## Catatan Milestone 2

### Ringkasan Evaluasi Kualitas Data
1. **Missing Values (Kelengkapan Data):**
   - Dataset APBD Pemda Indonesia ini tidak memiliki *missing values* (0 null) pada seluruh 11 kolom.
   - Semua field kunci (`tahun`, `bulan`, `id_provinsi`, `id_pemda`, `no_akun`) serta nilai numerik (`anggaran`, `realisasi`, `persentase`) terisi dengan lengkap.

2. **Deduplikasi (Keunikan Data):**
   - Pengujian keunikan baris menggunakan kunci komposit bisnis: `['tahun', 'bulan', 'id_provinsi', 'id_pemda', 'no_akun']`.
   - Dari total **3.132.863 baris**, terdapat **3.132.863 baris unik** (0 duplikat).

3. **Penanganan Outlier & Anomali Nilai:**
   - Metode IQR statistik standar tidak diterapkan secara langsung pada nilai nominal `anggaran` dan `realisasi` karena sifat hirarkis laporan APBD (perbedaan skala yang wajar antara akun utama ber-skala triliunan rupiah dengan sub-akun berskala jutaan rupiah).
   - Sebagai gantinya, dilakukan *domain constraint validation* dengan clipping nilai `anggaran` dan `realisasi` agar tidak bernilai negatif, serta pembatasan rasio `persentase` realisasi pada batas wajar (0% hingga 200%).

4. **Hasil Akhir Pipeline Cleaning:**
   - **Ukuran Sebelum:** 3.132.863 baris
   - **Ukuran Sesudah:** 3.132.863 baris
   - **Status:** Data bersih dan terverifikasi untuk dilanjutkan ke **Milestone 3 (Exploratory Data Analysis / Feature Engineering)**.